In [ ]:
# MAE based on the Plant Organelle Segmentation Dataset,
# which is a dataset for 2D segmentation of plant organelles. 
# The dataset contains 2D images of plant cells with annotated organelles, 
# including nuclei, mitochondria, and chloroplasts.
# All the 2D images are individually imaged from difference part of plant cell, not from the same 3D volume.
# The dataset is used to train and evaluate the performance of the MAE、Dinov3 model 
# for 2D segmentation tasks.

Download the models first

In [ ]:
import os
from huggingface_hub import snapshot_download
from pathlib import Path
BASE_DIR = Path.cwd().resolve().parent

local_model_dir = (BASE_DIR / "models")
snapshot_download(repo_id="Zeyu0102/EMCFsys_models", repo_type="model",
                  local_dir=local_model_dir,
                  local_dir_use_symlinks=False, resume_download=True,
                  max_workers=1)


In [ ]:
# Set the root path
# When it is new kernel, 
# you need to run the following code to set the root path, 
# otherwise it will report an error that the module cannot be found.
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory

import os
from huggingface_hub import snapshot_download
BASE_DIR = pathlib.Path.cwd().resolve().parent
local_dir = str(BASE_DIR / "demo" /"datasets_temp" / "EMCFsys_PlantOrgSegDataset")
# save the dataset to local_dir, and set resume_download=True to resume the download if it was interrupted
snapshot_download(repo_id="Zeyu0102/EMCFsys_PlantOrgSegDataset", repo_type="dataset",
                  local_dir=local_dir,
                  local_dir_use_symlinks=False, 
                  resume_download=True)

print("Plant Organelle segmentation Dataset downloaded successfully.")

#### finetune the MAE model on the Plant Organelle Segmentation Dataset

To get better segmentation result, we finetune the backbone with low lr,
that is the backbone's lr was set to 0.1x of base_lr, and the decoder_head's lr was set to 10x of base_lr.



In [ ]:
# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------

from mmengine import Config
cfg_path =str(BASE_DIR / "configs" / "SemanticSegmentation" / "MAE_EMCF_UPerNet.py")
cfg = Config.fromfile(cfg_path)

# set the path to the pretrained model from the huggingface hub, 
# you can change it to your own path if you have downloaded the model to a different location
# models are named as "xxxx_backbone_mmbackend.pth" in the huggingface hub
model_path = str(BASE_DIR / "models" / "MAE_backbone_mmbackend.pth")
cfg.load_from = model_path

# set the path to the dataset where the dataset is unzipped, 
# you can change it to your own path if you have downloaded the dataset to a different location
data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)

# set the working directory to save logs and models
work_dir = str(BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet")
cfg.work_dir = work_dir

# check the batch size of the training dataloader, if it is too large, you can reduce it to avoid out of memory error
cfg.train_dataloader.batch_size = 8

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()

#### Get test metric
Set config and checkpoint path

In [ ]:
import pathlib
import sys
import torch
import cv2
import matplotlib.pyplot as plt
from mmengine import Config

# ===================== Base Path Dir=====================
# BASE_DIR = pathlib.Path(r"Fill your absolute path").resolve()
BASE_DIR = pathlib.Path.cwd().resolve().parent

sys.path.insert(0, str(BASE_DIR))
# set the cfg_path and model_path

cfg_path = str(BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet" / "MAE_EMCF_UPerNet.py")
# get the path to the best checkpoint, using search to find the best checkpoint in the work_dir
for file in (BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet").glob("best_mIoU_epoch_*.pth"):
    model_path = str(file)
    break
# =================================================================

# 1. load the config and model
cfg = Config.fromfile(cfg_path)
cfg.load_from = model_path

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.test()


#### Finetune the DinoV3 model

In [ ]:
# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------


from mmengine import Config
cfg_path =str(BASE_DIR / "demo" / "configs" / "SemanticSegmentation" / "DinoV3_UperNet.py")
cfg = cfg = Config.fromfile(cfg_path)

model_path = str(BASE_DIR / "models" / "Dinov3_backbone_mmbackend.pth")
data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
work_dir = str(BASE_DIR / "demo" / "logs" / "DinoV3_UperNet")

cfg.load_from = model_path
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)
cfg.work_dir = work_dir
from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()

#### Training from Scratch

In [ ]:
# Train from Scratch, you can set cfg.load_from = None to train from scratch, 
# but it will take a long time to train and the performance may not be as good as using a pretrained model.

# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------

from mmengine import Config
cfg_path =str(BASE_DIR / "configs" / "SemanticSegmentation" / "MAE_EMCF_UPerNet.py")
cfg = Config.fromfile(cfg_path)

# set the path to the pretrained model from the huggingface hub, 
# you can change it to your own path if you have downloaded the model to a different location
# models are named as "xxxx_backbone_mmbackend.pth" in the huggingface hub


# set cfg.load_from = None to train from scratch
cfg.load_from = None

# set lr to 0.001, and set the backbone, decode_head, and auxiliary_head to have the same learning rate as the optimizer
cfg.optim_wrapper = dict(
    clip_grad=dict(max_norm=1.0),
    optimizer=dict(lr=0.001, type='AdamW', weight_decay=0.01),
    paramwise_cfg=dict(
        bias_decay_mult=0.0,
        custom_keys=dict(
            auxiliary_head=dict(lr_mult=1.0),
            backbone=dict(lr_mult=1.0),
            decode_head=dict(lr_mult=1.0)),
        norm_decay_mult=0.0),
    type='OptimWrapper')

data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)


work_dir = str(BASE_DIR / "demo" / "logs" / "Scratch_UperNet_plantseg")
cfg.work_dir = work_dir

cfg.train_dataloader.batch_size = 8

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()

## Train the model using less data

EMCFsys model (MAE)： 80% data

In [ ]:
# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------

from mmengine import Config
cfg_path =str(BASE_DIR / "configs" / "SemanticSegmentation" / "MAE_EMCF_UPerNet.py")
cfg = Config.fromfile(cfg_path)

# set the path to the pretrained model from the huggingface hub, 
# you can change it to your own path if you have downloaded the model to a different location
# models are named as "xxxx_backbone_mmbackend.pth" in the huggingface hub
model_path = str(BASE_DIR / "models" / "MAE_backbone_mmbackend.pth")
cfg.load_from = model_path

# set the path to the dataset where the dataset is unzipped, 
# you can change it to your own path if you have downloaded the dataset to a different location
data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)

# set the working directory to save logs and models
work_dir = str(BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet_80_percent")
cfg.work_dir = work_dir

# using 80% of training data
train_split_dir = str("splits/train_80.txt")
cfg.train_dataloader.dataset.ann_file = train_split_dir
# check the batch size of the training dataloader, if it is too large, you can reduce it to avoid out of memory error
cfg.train_dataloader.batch_size = 8

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()

EMCFsys model (MAE)： 40% data

In [ ]:
# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------

from mmengine import Config
cfg_path =str(BASE_DIR / "configs" / "SemanticSegmentation" / "MAE_EMCF_UPerNet.py")
cfg = Config.fromfile(cfg_path)

# set the path to the pretrained model from the huggingface hub, 
# you can change it to your own path if you have downloaded the model to a different location
# models are named as "xxxx_backbone_mmbackend.pth" in the huggingface hub
model_path = str(BASE_DIR / "models" / "MAE_backbone_mmbackend.pth")
cfg.load_from = model_path

# set the path to the dataset where the dataset is unzipped, 
# you can change it to your own path if you have downloaded the dataset to a different location
data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)

# set the working directory to save logs and models
work_dir = str(BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet_40_percent")
cfg.work_dir = work_dir

# using 80% of training data
train_split_dir = str("splits/train_40.txt")
cfg.train_dataloader.dataset.ann_file = train_split_dir
# check the batch size of the training dataloader, if it is too large, you can reduce it to avoid out of memory error
cfg.train_dataloader.batch_size = 8

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()

EMCFsys model (MAE)： 20% data

In [ ]:
# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------

from mmengine import Config
cfg_path =str(BASE_DIR / "configs" / "SemanticSegmentation" / "MAE_EMCF_UPerNet.py")
cfg = Config.fromfile(cfg_path)

# set the path to the pretrained model from the huggingface hub, 
# you can change it to your own path if you have downloaded the model to a different location
# models are named as "xxxx_backbone_mmbackend.pth" in the huggingface hub
model_path = str(BASE_DIR / "models" / "MAE_backbone_mmbackend.pth")
cfg.load_from = model_path

# set the path to the dataset where the dataset is unzipped, 
# you can change it to your own path if you have downloaded the dataset to a different location
data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)

# set the working directory to save logs and models
work_dir = str(BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet_20_percent")
cfg.work_dir = work_dir

# using 20% of training data
train_split_dir = str("splits/train_20.txt")
cfg.train_dataloader.dataset.ann_file = train_split_dir
# check the batch size of the training dataloader, if it is too large, you can reduce it to avoid out of memory error
cfg.train_dataloader.batch_size = 8

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()

EMCFsys model (MAE)： 10% data

In [ ]:
# When it is new kernel, you need to run the following code to set the root path, otherwise it will report an error that the module cannot be found.
#------------------- Set the root path-------------------------
from mmengine import Config
import pathlib
import sys
# Get the base directory of the project, which is the parent directory of the current working directory
BASE_DIR = pathlib.Path.cwd().resolve().parent
print(f"BASE_DIR: {BASE_DIR}")
# set the root directory of the project to the module search path
root = pathlib.Path(BASE_DIR)
# add the root directory to the module search path
sys.path.insert(0, str(root))
#-----------------------------------------------------

from mmengine import Config
cfg_path =str(BASE_DIR / "configs" / "SemanticSegmentation" / "MAE_EMCF_UPerNet.py")
cfg = Config.fromfile(cfg_path)

# set the path to the pretrained model from the huggingface hub, 
# you can change it to your own path if you have downloaded the model to a different location
# models are named as "xxxx_backbone_mmbackend.pth" in the huggingface hub
model_path = str(BASE_DIR / "models" / "MAE_backbone_mmbackend.pth")
cfg.load_from = model_path

# set the path to the dataset where the dataset is unzipped, 
# you can change it to your own path if you have downloaded the dataset to a different location
data_root = str(BASE_DIR / "demo" / "datasets_temp" / "EMCFsys_PlantOrgSegDataset")
cfg.data_root = data_root
from configs.register_models.runtime import set_data_root
set_data_root(cfg, cfg.data_root)

# set the working directory to save logs and models
work_dir = str(BASE_DIR / "demo" / "logs" / "MAE_EMCF_UPerNet_10_percent")
cfg.work_dir = work_dir

# using 10% of training data
train_split_dir = str("splits/train_10.txt")
cfg.train_dataloader.dataset.ann_file = train_split_dir
# check the batch size of the training dataloader, if it is too large, you can reduce it to avoid out of memory error
cfg.train_dataloader.batch_size = 8

from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
# start training
runner.train()